<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Lab 3: Manufacturing Signals and<br>Feature Engineering</h1>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/labs/week03/lab03_manufacturing_signals_and_feature_engineering.ipynb)

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1) · [Lab Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Lab_index.ipynb)

**ME 5995 — AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science**  
**Wayne State University**

**Graded individual Lab · Approximately 95 minutes including submission**  
**Version:** Student Notebook

## Student Information

Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Overview and learning objectives

**Workflow:** raw signal -> time/features -> FFT/STFT -> feature table ->
relevance/redundancy -> three-feature subset -> guided ML and sound previews.
After this Lab you can:

- construct physical time and identify sensor units and cutter/cut grouping;
- compute mean, sample SD, RMS and peak-to-peak and interpret force/vibration differences;
- organize one cut's feature vector as a table row;
- interpret whole-record FFT amplitude and time-localized STFT PSD;
- use Pearson and redundancy evidence to justify an exploratory feature subset;
- separate predictors, target and groups for later regression/validation;
- explain a guided linear-frequency versus Mel manufacturing-sound example.

Prerequisites: Week 2 NumPy/pandas/Matplotlib and Week 3 lecture concepts.
Imports, loading, spectral helpers and plotting templates are supplied.

**Work pattern:** run supplied code, replace REQUIRED `None` placeholders, and
write short responses for seven checkpoints. `None` prints an unfinished reminder;
an error-free starter run is not a completed submission. Rerun from the top after
edits. Parts 9–10 are ungraded; optional work is not needed for full credit.

| Parts | Minutes |
|---|---:|
| 1 / 2 / 3: time, features, three cuts | 8 / 15 / 9 |
| 4 / 5 / 6: FFT, STFT, table meaning | 10 / 11 / 3 |
| 7 / 8 / 9: full table, selection, guided X/y/groups | 9 / 11 / 3 |
| 10: guided sound | 6 |
| Restart, check and submit | 10 |
| Total | 95 (85 activity + 10 submission) |

Use Colab or the designated local notebook environment. Submit
`Lab03_Firstname_Lastname.ipynb` and `.pdf` through Canvas, with code, outputs,
figures and responses visible. No separate report; follow Canvas deadlines.

## Data provenance and engineering limits

PHM 2010: full c1 cuts 1/158/315; Force X (N), Vibration X (g); sampling 50 kHz,
Nyquist 25 kHz. These are sequence positions, not wear classes. Unequal durations
and unknown engagement phases prevent cutting-only or causal claims. AE-RMS (V)
is processed, not raw high-frequency AE. The master has 945 cuts across three
cutters, with 42 scalar sensor features.

The [PHM data card](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/README.md)
records rights limits: the mirror's CC0 label is not an official PHM license;
the repository MIT license does not override dataset rights.

MIMII public 1.0, Purohit et al., [DOI 10.5281/zenodo.3384388](https://doi.org/10.5281/zenodo.3384388),
CC BY-SA 4.0: source Channel 1, mono PCM16, 16 kHz, 10 seconds, no cropping,
resampling or normalization. The 0 dB condition includes factory noise. Our one
normal fan example comes from 220 course WAVs, not an official benchmark split.
Preserve attribution/ShareAlike; digital amplitude is not sound pressure.

In [ ]:
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen
import wave
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.precision', 4)  # Display only; calculations retain full precision.
# Wrap long feature names so required table columns remain visible in PDF output.
display(HTML('<style>table.dataframe {width:100%; table-layout:fixed;}'
             'table.dataframe th, table.dataframe td {white-space:normal;'
             'overflow-wrap:anywhere; font-size:10px; padding:4px;}</style>'))
BASE_URL = 'https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/'
DATA_PATHS = ['data/phm2010/c1_selected_cuts/c1_cut001.csv', 'data/phm2010/c1_selected_cuts/c1_cut158.csv', 'data/phm2010/c1_selected_cuts/c1_cut315.csv', 'data/phm2010/features/phm2010_features.csv', 'data/mimii/metadata.csv', 'data/mimii/audio/fan/id_00/normal/00000025.wav']

def read_public_bytes(relative_path):
    """Course-relative path in; bytes out; prefer checkout, else the same public URL."""
    for directory in (Path.cwd(), *Path.cwd().parents):
        candidate = directory / relative_path
        if candidate.is_file():
            return candidate.read_bytes()
    with urlopen(BASE_URL + relative_path, timeout=60) as response:
        return response.read()

def read_csv(relative_path):
    """Course CSV path in; DataFrame out; retain round-trip numeric precision."""
    return pd.read_csv(BytesIO(read_public_bytes(relative_path)), float_precision='round_trip')

# Shared course paths load the same complete, untrimmed records locally and in Colab.
raw = {cut: read_csv(path) for cut, path in zip((1, 158, 315), DATA_PATHS[:3])}
# Each row holds simultaneous sensor samples; channel suffixes identify N, g and V.
expected_channels = ['force_x_N','force_y_N','force_z_N','vibration_x_g',
                     'vibration_y_g','vibration_z_g','ae_rms_V']
for cut, frame in raw.items():
    if list(frame.columns) != expected_channels or not np.isfinite(frame.to_numpy()).all():
        raise ValueError(f'Unexpected channel schema or nonfinite samples for cut {cut}')
print('Loaded c1 cuts:', list(raw))

## [GUIDED] Supplied representation helpers

**Run unchanged; do not study this implementation line by line.** FFT returns Hz
and signal-unit amplitude; STFT returns Hz, frame-center seconds and one-sided PSD.
Call templates are provided later, including Mel. No syntax memorization.

In [ ]:
def checked(x):
    """Signal in; finite 1-D float array out; reject invalid records."""
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 1 or len(x) < 2 or not np.isfinite(x).all():
        raise ValueError('Expected a finite one-dimensional signal with at least two samples')
    return x

def hann(n):
    """Length in; periodic Hann weights out; taper finite-record boundaries."""
    return 0.5 - 0.5*np.cos(2*np.pi*np.arange(n)/n)

def one_sided_factors(n):
    """Signal length in; one-sided factors out; preserve DC/Nyquist bins."""
    factors = np.full(n//2+1, 2.0)
    factors[0] = 1
    if n % 2 == 0:
        factors[-1] = 1
    return factors

def fft_amplitude(x, fs):
    """Signal and fs in; Hz/amplitude out; mean-removed Hann, no padding."""
    x = checked(x)
    if fs <= 0:
        raise ValueError('Positive sampling rate required')
    # Hann tapering reduces leakage from finite-record boundaries.
    window = hann(len(x))
    # Remove the mean; real-signal rFFT keeps nonnegative frequencies.
    # One-sided factors and window gain retain the sensor amplitude unit.
    amplitude = np.abs(np.fft.rfft((x-x.mean())*window))*one_sided_factors(len(x))/window.sum()
    # Map frequency bins to Hz using sampling interval 1/fs.
    return np.fft.rfftfreq(len(x), 1/fs), amplitude

def stft_psd(x, fs, n_fft, hop):
    """Signal/fs/window/hop in; Hz/time/PSD out; global centering, complete frames."""
    x = checked(x)
    if fs <= 0 or not 2 <= n_fft <= len(x) or not 1 <= hop <= n_fft:
        raise ValueError('Invalid STFT parameters')
    # Remove one record-wide offset so local spectral changes remain comparable.
    centered = x-x.mean()
    # Overlap windows to retain time location; hop is the advance in samples.
    frames = np.lib.stride_tricks.sliding_window_view(centered, n_fft)[::hop]
    window = hann(n_fft)
    spectrum = np.fft.rfft(frames*window, axis=1)
    # Normalize to one-sided power density: signal-unit squared per Hz.
    psd = abs(spectrum)**2/(fs*np.sum(window**2))*one_sided_factors(n_fft)
    # Locate each local spectrum at its window center, unlike a whole-record FFT.
    times = (np.arange(len(frames))*hop+n_fft/2)/fs
    return np.fft.rfftfreq(n_fft, 1/fs), times, psd.T

def hz_to_mel(hz):
    """Hz in; HTK Mel values out; use the stated auditory frequency mapping."""
    return 2595*np.log10(1+np.asarray(hz)/700)

def mel_to_hz(mel):
    """HTK Mel values in; Hz out; invert the same frequency mapping."""
    return 700*(10**(np.asarray(mel)/2595)-1)

def mel_filters(fs, n_fft, n_mels):
    """fs/FFT/band counts in; centers/weights out; peak-one HTK triangles, no area norm."""
    if fs <= 0 or n_fft < 2 or n_mels < 1:
        raise ValueError('Invalid Mel parameters')
    # Equal Mel steps give unequal Hz bandwidths across the sound spectrum.
    edges = mel_to_hz(np.linspace(0, hz_to_mel(fs/2), n_mels+2))
    frequencies = np.fft.rfftfreq(n_fft, 1/fs)
    weights = np.maximum(0, np.minimum(
        (frequencies[None, :]-edges[:-2, None])/(edges[1:-1]-edges[:-2])[:, None],
        (edges[2:, None]-frequencies[None, :])/(edges[2:]-edges[1:-1])[:, None]))
    if np.any(weights.sum(axis=1) == 0):
        raise ValueError('Empty Mel bands; increase FFT length or reduce band count')
    return edges[1:-1], weights

def mel_spectrogram(psd, fs, n_fft, n_mels):
    """PSD and band settings in; centers/band power out; integrate weighted density."""
    centers, weights = mel_filters(fs, n_fft, n_mels)
    # PSD times bin width gives integrated band power, not another density.
    return centers, weights @ psd * (fs/n_fft)

def power_db(power, reference=1.0, floor=1e-20):
    """Power/reference in; dB out; floor zeros for display, not feature calculation."""
    power = np.asarray(power)
    if reference <= 0 or floor <= 0 or np.any(power < 0) or not np.isfinite(power).all():
        raise ValueError('Expected finite nonnegative power and positive reference/floor')
    return 10*np.log10(np.maximum(power, floor)/reference)

def envelope(x, fs, bins=1800):
    """Signal/fs in; time/min/max out; display pooling only, no analysis resampling."""
    edges = np.linspace(0, len(x), min(bins, len(x))+1, dtype=int)
    return ((edges[:-1]+edges[1:]-1)/(2*fs),
            np.array([x[a:b].min() for a, b in zip(edges[:-1], edges[1:])]),
            np.array([x[a:b].max() for a, b in zip(edges[:-1], edges[1:])]))

# Part 1 — Load and Understand a Manufacturing Signal

**Time guide: 8 minutes.**

In [ ]:
record = raw[158]
print('Shape:', record.shape)
print('Channels:', list(record.columns))
display(record.head(3))

One CSV row is one simultaneous sample of seven channels. For N samples,
`t[n] = n/fs`, nominal duration is N/fs, and the last sample is at (N-1)/fs.
At 50 kHz, the sample interval is 20 microseconds and Nyquist is 25 kHz.
The Nyquist limit is $f_{\mathrm{Nyquist}}=f_s/2$; components above it can alias
into lower frequencies. This is not evidence of an aliasing event in PHM.
The detailed synthetic comparison stays in lecture. Use `np.arange(N)` for sample indices.

## [REQUIRED CHECKPOINT 1] Physical time axis and labeled waveform

In [ ]:
fs = None  # TODO: Set the documented PHM sampling rate in Hz.
N = len(record)
# Sample index becomes elapsed time; adjacent samples are 1/fs seconds apart.
time_s = None  # TODO: Convert sample indices to elapsed seconds using fs.
duration_s = None  # TODO: Convert the full sample count to duration in seconds.
if fs is None or time_s is None or duration_s is None:
    print('Complete the time-axis inputs in Checkpoint 1, then rerun.')
else:
    print('fs (Hz), N, duration (s), last sample (s):', fs, N, duration_s, time_s[-1])
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(time_s, record['vibration_x_g'], linewidth=0.3)
    ax.set(title='c1 cut 158: Vibration X', xlabel='Time (s)', ylabel='Vibration X (g)')
    fig.tight_layout(); plt.show()

### Checkpoint 1 response (1–3 sentences)

What does a raw CSV row represent, and why is the last sample time before N/fs?

**TODO:** Write your response here.

# Part 2 — Time-Domain Feature Extraction

**Time guide: 15 minutes.**

| Feature | Definition | Meaning |
|---|---|---|
| Mean | mean(x) | Signed average level |
| Sample SD | std(x, ddof=1) | Fluctuation about the mean |
| RMS | sqrt(mean(x squared)) | Total magnitude including mean |
| Peak-to-peak | max(x)-min(x) | Observed range; sensitive to extremes |

For $N$ original samples $x_i$, the definitions are:
$$\bar{x}=\frac{1}{N}\sum_{i=1}^{N}x_i,
\qquad s=\sqrt{\frac{1}{N-1}\sum_{i=1}^{N}(x_i-\bar{x})^2}$$
$$x_{\mathrm{RMS}}=\sqrt{\frac{1}{N}\sum_{i=1}^{N}x_i^2},
\qquad x_{p-p}=x_{\max}-x_{\min}$$
All retain the sensor unit. The exact finite-sample relationship is
$$x_{\mathrm{RMS}}^2=\bar{x}^2+\frac{N-1}{N}s^2.$$
Use this to compare mean load and fluctuations in force/vibration. Do not center
before raw RMS; no derivation is required. Synthetic syntax example:

In [ ]:
example = np.array([1., 2., 3.])
print('Synthetic mean / sample SD:', np.mean(example), np.std(example, ddof=1))
print('Synthetic squared values:', example**2)

## [REQUIRED CHECKPOINT 2] Four features for Force X and Vibration X

In [ ]:
def core_features(x):
    """One full signal in; four same-unit scalars out (or None until completed)."""
    x = np.asarray(x, dtype=float)
    # Mean is the average signed sensor level over the full record.
    mean = None  # TODO: Average the signed samples.
    # Sample SD describes fluctuations around that average level.
    sd = None  # TODO: Compute sample SD with ddof=1.
    # Raw RMS includes fluctuations and any nonzero mean component.
    rms = None  # TODO: Root-mean-square the original samples.
    # The observed amplitude range can be sensitive to isolated extremes.
    peak_to_peak = None  # TODO: Find the full observed amplitude range.
    if any(value is None for value in (mean, sd, rms, peak_to_peak)):
        return None
    return dict(mean=mean, sd=sd, rms=rms, peak_to_peak=peak_to_peak)

# Check whether mean load matters for force and whether vibration is near zero mean.
force_features = core_features(record['force_x_N'])
vibration_features = core_features(record['vibration_x_g'])
core_table = None
if force_features is None or vibration_features is None:
    print('Complete the four expressions in Checkpoint 2.')
else:
    core_table = pd.DataFrame([force_features, vibration_features],
                             index=['Force X (N)', 'Vibration X (g)'])
    display(core_table)

### Checkpoint 2 response (1–3 sentences)

Explain the features physically and the difference between Force X and Vibration X SD/RMS.

**TODO:** Write your response here.

# Part 3 — Compare Early / Middle / Late Machining Records

**Time guide: 9 minutes.**

## [REQUIRED CHECKPOINT 3] Reuse the function for three cuts

In [ ]:
# Multiple scalar features from one complete cut become one table row with its identity.
comparison_rows = []
for cut, frame in raw.items():
    force = None  # TODO: Apply core_features to this cut's Force X channel.
    vibration = None  # TODO: Apply the same function to its Vibration X channel.
    if force is not None and vibration is not None:
        row = {'cutter_id': 'c1', 'cut_number': cut}
        row.update({'force_x_'+name: value for name, value in force.items()})
        row.update({'vibration_x_'+name: value for name, value in vibration.items()})
        comparison_rows.append(row)
comparison = pd.DataFrame(comparison_rows)
if len(comparison) == 3:
    display(comparison)
    print('All force features: N; all vibration features: g.')
else:
    print('Complete Checkpoints 2 and 3 to obtain three rows.')

### Checkpoint 3 response (1–3 sentences)

Describe one change across the cuts. What is one feature-table row, and what cannot these three records establish?

**TODO:** Write your response here.

# Part 4 — Frequency-Domain Analysis with FFT

**Time guide: 10 minutes.**

The helper uses full-record mean removal, periodic Hann, N-point rFFT,
no padding and coherent-gain corrected one-sided amplitude; peak search excludes
DC. Bin spacing is fs/N, not physical resolving power; windowing spreads energy
and off-bin amplitudes can be biased. Use cut 315 Vibration X for both FFT and STFT.

In [ ]:
vibration_late = raw[315]['vibration_x_g'].to_numpy()

## [REQUIRED CHECKPOINT 4] Whole-record amplitude spectrum

In [ ]:
fft_result = None  # TODO: Call fft_amplitude with late-cut vibration and fs.
if fft_result is None:
    print('Complete Checkpoint 1 and the FFT call.')
else:
    frequency_hz, amplitude_g = fft_result
    peak = 1 + np.argmax(amplitude_g[1:])
    print('Bin spacing (Hz):', fs/len(vibration_late))
    print('Non-DC peak (Hz, g):', frequency_hz[peak], amplitude_g[peak])
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(frequency_hz/1000, amplitude_g, linewidth=0.6)
    ax.set(xlabel='Frequency (kHz)', ylabel='Amplitude (g)', xlim=(0,25),
           title='c1 cut 315: Vibration X, full-record Hann FFT')
    fig.tight_layout(); plt.show()

### Checkpoint 4 response (1–3 sentences)

What prominent component is present? What timing information is unavailable, and is the peak automatically a useful wear predictor?

**TODO:** Write your response here.

# Part 5 — Time-Frequency Analysis with STFT

**Time guide: 11 minutes.**

![Conceptual FFT whole-record spectrum versus overlapping-window STFT](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab03/fft_vs_stft.png)

*Conceptual illustration: the upper-right FFT horizontal axis should read Frequency (Hz), not time (sec). The panels illustrate the workflow; they are not a matched PHM calculation.*

Use periodic Hann, 2048 samples, hop 512, overlap 1536 (75%), full-record
mean removal, complete frames only, no padding or per-frame detrending. PSD is
one-sided g²/Hz. At 50 kHz: window 40.96 ms, hop 10.24 ms, bin spacing 24.414 Hz.
These are fixed settings, not a tuning exercise. Frame centers label the time axis.
The display uses 10 log10(PSD / 1 g²/Hz), floor 1e-20 and an 80-dB color range.

## [REQUIRED CHECKPOINT 5] Spectrogram and FFT/STFT interpretation

In [ ]:
stft_result = None  # TODO: Call stft_psd with late-cut vibration, fs, window 2048 and hop 512.
if stft_result is None:
    print('Complete Checkpoint 1 and the STFT call.')
else:
    stft_hz, frame_s, psd = stft_result
    db = power_db(psd)
    fig, ax = plt.subplots(figsize=(8, 3.5))
    image = ax.pcolormesh(frame_s, stft_hz/1000, db, shading='auto',
                         vmin=db.max()-80, vmax=db.max(), rasterized=True)
    ax.set(xlabel='Frame-center time (s)', ylabel='Frequency (kHz)', ylim=(0,25),
           title='c1 cut 315: Vibration X STFT, 2048 / hop 512')
    fig.colorbar(image, ax=ax, label='PSD (dB re 1 g²/Hz)')
    fig.tight_layout(); plt.show()

### Checkpoint 5 response (1–3 sentences)

What does FFT retain and lose, and what does STFT add? Describe one visible time-dependent pattern without assigning a cutting phase or cause.

**TODO:** Write your response here.

# Part 6 — From Raw Signals to a Feature Table

**Time guide: 3 minutes.**

![Raw signal to a feature vector, a cut-level table row and later ML inputs](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab03/signal_to_features.png)

**[GUIDED; CP3 continuation]** One raw record -> four summaries on two
channels -> an eight-feature vector -> one row with a cutter/cut key.
Revisit CP3; no duplicate table or answer. Windows from one record are not
independent wear experiments.

# Part 7 — Explore the Full PHM Feature Dataset

**Time guide: 9 minutes.**

In [ ]:
# One master row represents one cutter/cut, not one sensor sample.
master = read_csv(DATA_PATHS[3])
# Sensor features are candidate inputs; cutter/cut identifiers remain metadata.
candidates = ['force_x_mean','force_x_sd','vibration_x_sd','vibration_x_rms','ae_rms_mean']
units = dict(zip(candidates, ['N','N','g','g','V']))
if master.shape != (945,56) or master.duplicated(['cutter_id','cut_number']).any():
    raise ValueError('Unexpected PHM master shape or duplicate cut keys')
print('Master shape:', master.shape)
# wear_mean_um is the continuous target; wear_level is a separate categorical target.
display(master[['cutter_id','cut_number',*candidates,'wear_mean_um','wear_level']].head(3))
display(pd.DataFrame({'Role': ['Metadata','Sensor predictors','Continuous target','Categorical target'],
 'Examples': ['cutter_id, cut_number, source_file', ', '.join(candidates),
              'wear_mean_um (micrometers)', 'wear_level (course-defined category)']}))

Use wear_mean_um as the continuous target. Wear level is categorical despite
integer storage; do not use it in Pearson correlation. Flute wear and wear_max_um
also leak target information if included as predictors. Cut number is sequence,
not a default predictor. AE-RMS mean is average processed signal level (V), and
AE-RMS SD is its fluctuation (V); neither is a raw AE frequency measurement.

For paired feature values $x_i$ and mean wear $y_i$ across cuts:
$$r_{xy}=\frac{\sum_{i=1}^{N}(x_i-\bar{x})(y_i-\bar{y})}
{\sqrt{\sum_{i=1}^{N}(x_i-\bar{x})^2}\sqrt{\sum_{i=1}^{N}(y_i-\bar{y})^2}}.$$

- $r$ describes the strength and direction of a linear association.
- Correlation does not imply causation or guaranteed predictive value.

## [REQUIRED CHECKPOINT 6] Feature–wear relationship and Pearson correlation

Run this supplied plot for **force_x_mean versus wear_mean_um**. Then
complete one Pearson calculation and interpret the plot and r together.
Pearson measures linear association, not causation. Low r does not rule out
nonlinear or cutter-dependent information.

In [ ]:
feature = 'force_x_mean'
fig, ax = plt.subplots(figsize=(7, 3.5))
for cutter, frame in master.groupby('cutter_id'):
    ax.scatter(frame['wear_mean_um'], frame[feature], s=8, alpha=0.6, label=cutter)
ax.set(xlabel='Mean wear (µm)', ylabel='Force X mean (N)',
       title='Force X mean versus wear, by cutter')
ax.legend(); fig.tight_layout(); plt.show()

**Syntax:** `table['feature'].corr(table['target'], method='pearson')` returns one r.

In [ ]:
# Pearson r describes linear wear association, not causation or predictive performance.
target_r = None  # TODO: Pearson r between master['force_x_mean'] and master['wear_mean_um']
if target_r is None:
    print('Complete the Pearson expression for Checkpoint 6.')
else:
    print('Force X mean versus wear, Pearson r:', target_r)

### Checkpoint 6 response (1–2 sentences)

Describe the force–wear association shown by the scatter plot and r. State one limitation of this pooled relationship.

**TODO:** Write your response here.

# Part 8 — Feature Relevance, Redundancy, and Simple Selection

**Time guide: 11 minutes.**

| Relevance | Redundancy | Engineering meaning |
|---|---|---|
| Related to the target? | Repeats another feature's information? | Meaningful physical quantity? |

**Relevance + redundancy + engineering interpretation → compact feature subset.**

| Method type | Basic idea | Examples | Lab 3 |
|---|---|---|---|
| Filter | Evaluate before model training | Correlation, variance, mutual information | Pearson/manual reasoning used |
| Wrapper | Compare subsets by repeated model training | Forward selection, RFE | Concept only |
| Embedded | Selection during model training | L1/Lasso, tree importance | Concept only |

Guided lecture recap only; no additional implementation.

## [REQUIRED CHECKPOINT 7] Identify redundancy and select three features

Run the supplied summary and heatmap. Inspect one redundant pair, choose
**three** features, and justify the list in 2–3 sentences. No correlation-matrix
coding, ranking or significance tests are required. This is a simple filter-style
exercise; wrapper/embedded methods remain lecture context.

In [ ]:
# Supplied summary: fixed order, not a ranking or an additional task.
# Feature-target correlation supplies relevance evidence, not an optimal ranking.
correlation_summary = master[candidates].corrwith(master['wear_mean_um'], method='pearson')
display(correlation_summary.rename('Pearson r with wear').to_frame())
by_cutter = {cutter: frame['force_x_mean'].corr(frame['wear_mean_um'])
             for cutter, frame in master.groupby('cutter_id')}
display(pd.Series(by_cutter, name='Guided comparison: Force X mean r within cutter'))
# Feature-feature correlation supplies redundancy evidence for the same candidates.
feature_corr = master[candidates].corr(method='pearson')
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(feature_corr, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(5), candidates, rotation=45, ha='right')
ax.set_yticks(range(5), candidates)
ax.set_title('Five candidates: Pearson correlation')
for i in range(5):
    for j in range(5): ax.text(j, i, f'{feature_corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=9)
fig.colorbar(image, ax=ax, label='Pearson r (dimensionless)')
fig.tight_layout(); plt.show()

In [ ]:
# For near-zero-mean vibration, SD and RMS can carry nearly redundant information.
fig, ax = plt.subplots(figsize=(5, 3.5))
for cutter, frame in master.groupby('cutter_id'):
    ax.scatter(frame['vibration_x_sd'], frame['vibration_x_rms'], s=8, alpha=0.5, label=cutter)
ax.set(xlabel='Vibration X sample SD (g)', ylabel='Vibration X raw RMS (g)',
       title='Redundancy check: SD versus RMS')
ax.legend(); fig.tight_layout(); plt.show()

Choose exactly **three distinct candidate names**. Consider relevance,
redundancy and physical meaning; there is no unique correct subset.

**This is exploratory feature selection, not an optimized feature-selection
procedure.** The 945 cuts are three related trajectories. Full-data inspection
does not establish independent test performance; later data-dependent selection
and preprocessing must use training data only.

In [ ]:
# Combine relevance, redundancy and sensor meaning; no unique list is required.
selected_features = None  # TODO: list of exactly three distinct candidate names
print('Selected:', selected_features)

### Checkpoint 7 response (2–3 sentences)

Identify a redundant pair and justify your retained/omitted features using relevance, redundancy and engineering meaning. State the exploratory limitation.

**TODO:** Write your response here.

# Part 9 — Guided Preview: X, y, and groups

**Time guide: 3 minutes.**

**[GUIDED; UNGRADED]** Run unchanged after selecting three features; inspect
the shapes and columns. X and y preview Week 4 regression; cutter groups preview
Week 6 group-aware validation. No extra answer, splitting or model training.

In [ ]:
X = y = groups = None
row_keys = master[['cutter_id','cut_number']].copy()
if selected_features is None:
    print('Select three features in Checkpoint 7, then rerun this guided preview.')
else:
    valid_selection = (len(selected_features) == 3 and
                       len(set(selected_features)) == len(selected_features) and
                       set(selected_features).issubset(candidates))
    if not valid_selection:
        raise ValueError('Select exactly three distinct candidate features')
    # X contains the selected sensor predictors, without wear-derived columns.
    X = master[selected_features].copy()
    # y is the continuous tool-wear target for Week 4 regression.
    y = master['wear_mean_um'].copy()
    # groups preserves cutter identity for later group-aware validation.
    groups = master['cutter_id'].copy()
    aligned = X.index.equals(y.index) and X.index.equals(groups.index) and X.index.equals(row_keys.index)
    if not valid_selection or not aligned or len(X) != 945 or list(X.columns) != selected_features:
        raise ValueError('Check candidate names, compactness and row alignment')
    if not np.isfinite(X.to_numpy()).all() or not np.isfinite(y.to_numpy()).all():
        raise ValueError('Nonfinite predictor/target values require investigation')
    print('X / y / groups shapes:', X.shape, y.shape, groups.shape)
    print('X columns:', list(X.columns), '| groups:', sorted(groups.unique()))
    display(pd.concat([row_keys, X, y], axis=1).head(3))

# Part 10 — Guided MIMII Sound Representation

**Time guide: 6 minutes.**

**[GUIDED; UNGRADED]** Run unchanged: normal fan id_00/00000025.wav;
1024-sample Hann, hop 256, 64 HTK Mel bands, 0–8 kHz (64-ms window, 16-ms hop).
These are Week 3 visualization settings, not final Week 10 NN parameters.

PCM16/32768 is fixed full-scale conversion, not normalization. Mel integrates PSD
through peak-one triangles without area normalization: wider bands collect more
power. Log-Mel reveals weaker bands. PSD density and band-power colors have
different units; compare the representations, not their color values.

**Linear spectrogram:** frequency directly in Hz. **Mel:** STFT information
aggregated into Mel-spaced bands. Mel is common for sound, not automatically
preferred for force or vibration.

In [ ]:
audio_metadata = read_csv(DATA_PATHS[4])
fan_meta = audio_metadata[(audio_metadata['machine_type']=='fan') &
                          (audio_metadata['machine_id']=='id_00') &
                          (audio_metadata['condition']=='normal') &
                          audio_metadata['classification_included']].sort_values(['source_file','recording_id']).iloc[0]
if '00000025' not in fan_meta['recording_id']:
    raise ValueError('Unexpected representative fan identity')
print('Representative:', fan_meta['recording_id'])
with wave.open(BytesIO(read_public_bytes(DATA_PATHS[5])), 'rb') as wav:
    audio_fs = wav.getframerate()
    if (wav.getnchannels(), wav.getsampwidth(), audio_fs, wav.getnframes()) != (1,2,16000,160000):
        raise ValueError('Unexpected audio format')
    # Convert signed PCM16 to floating full-scale amplitude, without peak normalization.
    audio = np.frombuffer(wav.readframes(wav.getnframes()), dtype='<i2').astype(float)/32768
# STFT localizes the fan frequency content in time using the supplied sound settings.
audio_hz, audio_s, audio_psd = stft_psd(audio, audio_fs, n_fft=1024, hop=256)
# Mel aggregates sound bands; it is not automatically preferred for force/vibration.
mel_centers, mel_power = mel_spectrogram(audio_psd, audio_fs, n_fft=1024, n_mels=64)
fig, axes = plt.subplots(4, 1, figsize=(8, 10))
t, low, high = envelope(audio, audio_fs)
axes[0].fill_between(t, low, high)
axes[0].set(title='Normal fan id_00 / 00000025.wav: display envelope',
            xlabel='Time (s)', ylabel='Amplitude (FS)')
# Log power reveals weaker structure; PSD density and band power have different units.
for ax, values, vertical, label, title in [
    (axes[1], power_db(audio_psd), audio_hz/1000, 'PSD (dB re 1 FS²/Hz)', 'Linear-frequency STFT'),
    (axes[2], mel_power, np.arange(64), 'Band power (FS²)', 'Mel-spectrogram'),
    (axes[3], power_db(mel_power), np.arange(64), 'Band power (dB re 1 FS²)', 'Log-Mel-spectrogram')]:
    limits = {} if ax is axes[2] else {'vmin': values.max()-80, 'vmax': values.max()}
    image = ax.pcolormesh(audio_s, vertical, values, shading='auto', rasterized=True, **limits)
    ax.set(title=title, xlabel='Frame-center time (s)',
           ylabel='Frequency (kHz)' if ax is axes[1] else 'Mel band index')
    if ax is axes[1]: ax.set_ylim(0,8)
    fig.colorbar(image, ax=ax, label=label)
fig.tight_layout(); plt.show()

### Guided interpretation (1–3 sentences; ungraded)

How do linear-frequency and Mel representations differ? Why is Mel common for
sound, and why is it not automatically appropriate for force/vibration?

**TODO:** Write a short comparison.

# Optional Challenge — choose one only if time permits

Ungraded options: another PHM channel/AE-RMS comparison; Pearson versus Spearman
(`method='spearman'`); or another MIMII machine type. You may explore raw crest
factor max(abs(x))/RMS or Pearson kurtosis mean((x-mean)^4)/mean((x-mean)^2)^2 on a
nonconstant record (no bias correction). Do not train models or create splits.

# Submission checklist

Rendered notebook checkboxes are not clickable. To record completion, edit this
Markdown cell and change `[ ]` to `[x]`.

- [ ] Student Information uses name and WSU AccessID.
- [ ] Seven checkpoints contain completed code, required outputs and short responses.
- [ ] CP1 waveform; CP2 feature table; CP3 three-cut table; CP4 FFT; CP5 STFT are visible.
- [ ] CP6 one scatter plot, one Pearson r and short interpretation are visible.
- [ ] CP7 supplied heatmap/redundancy plot, three-feature list and justification are visible.
- [ ] Run the ungraded X/y/groups preview and inspect its shapes; no extra answer.
- [ ] Guided sound example and short comparison are complete; optional work is not needed.
- [ ] Restart runtime/kernel and Run all after completing placeholders; resolve errors
  and all unfinished-checkpoint reminders. Save outputs without printing full raw data.
- [ ] Export a PDF and inspect code, tables, figures and responses for clipped content.
- [ ] Submit Lab03_Firstname_Lastname.ipynb and Lab03_Firstname_Lastname.pdf through Canvas.

No separate report. Canvas controls deadlines and policies.

# References

- [PHM Society 2010 challenge](https://phmsociety.org/phm_competition/2010-phm-society-conference-data-challenge/)
- [Course PHM provenance and license discussion](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/README.md)
- [PHM feature dictionary and definitions](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/DATA_DICTIONARY.md)
- Purohit et al., *MIMII Dataset: Sound Dataset for Malfunctioning Industrial Machine Investigation and Inspection*, 2019: [paper](https://arxiv.org/abs/1909.09347), [dataset](https://doi.org/10.5281/zenodo.3384388), [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).
- [Course MIMII transformations and metadata](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/mimii/README.md)
- [NumPy FFT reference](https://numpy.org/doc/stable/reference/routines.fft.html)

The course-derived scalar and Mel representations are instructional choices,
not claims about official benchmark protocols.